In [29]:

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader


# 1. Select device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# 2. Transform images into tensors
transform = transforms.ToTensor()


# 3. Load training dataset
train_data = MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)


# 4. Load testing dataset
test_data = MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)


# 5. Create DataLoaders
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False
)


# 6. Create CNN Model
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Feature extraction layers
        self.features = nn.Sequential(

            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Classification layers
        self.classifier = nn.Sequential(

            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(),

            nn.Linear(64, 10)
        )

    # Forward propagation
    def forward(self, x):

        x = self.features(x)

        x = x.view(x.size(0), -1)

        x = self.classifier(x)

        return x


# 7. Initialize model
model = SimpleCNN().to(device)


# 8. Define loss function
loss_fn = nn.CrossEntropyLoss()


# 9. Define optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# 10. Train the model
epochs = 3

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(images)

        # Calculate loss
        loss = loss_fn(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {avg_loss:.4f}"
    )


# 11. Evaluate the model
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Get predictions
        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        # Count correct predictions
        correct += (predictions == labels).sum().item()

        # Count total images
        total += labels.size(0)


# 12. Calculate test accuracy
accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")


# 13. Save the trained model
torch.save(model.state_dict(), "simple_cnn_mnist.pth")

print("Model Saved Successfully!")


Using device: cpu
Epoch 1/3 | Loss: 0.3508
Epoch 2/3 | Loss: 0.1016
Epoch 3/3 | Loss: 0.0679
Test Accuracy: 97.92%
Model Saved Successfully!
